# Ion permeation and conductance

Counts ion permeation events through the PKD2 pore for one applied-field run and converts the net flux to a conductance.

Two counting planes are taken every frame from the protein:
- upper plane: mean z of the four L641 carbonyl O atoms (bottom of the selectivity filter)
- lower plane: mean z of the four I680 Cα atoms (below the lower gate)

An **inward** event is an ion that starts above the upper plane and later passes below the lower plane. An **outward** event is an ion that starts below the lower plane and later reaches the upper plane. A jump across the periodic boundary resets the state of that ion. The conductance is g = n q e / (t V) with n the net number of events in the analysis window and V = |E| ⟨L_z⟩.

In [ ]:
import numpy as np
import MDAnalysis as mda

# settings for one run
gro_file = "step7.gro"
xtc_file = "step7.xtc"
ion_name = "POT"          # POT, SOD or CAL
ion_charge = 1            # 2 for Ca2+
field_strength = 0.02     # |electric-field-z| in V/nm
direction = "in"          # "in" for a negative field, "out" for a positive field
start_time = 50           # analysis window starts here (ns)

selections = {"reference": "resid 643 and name CA", "upper": "resid 641 and name O", "lower": "resid 680 and name CA"}

In [ ]:
u = mda.Universe(gro_file, xtc_file)
groups = {key: u.select_atoms(selection) for key, selection in selections.items()}
ions = u.select_atoms(f"resname {ion_name}")
n_frames = len(u.trajectory)
ion_z = np.empty((n_frames, len(ions)))      # ion z relative to the D643 CA centroid (A)
upper_z = np.empty(n_frames)
lower_z = np.empty(n_frames)
box_z = np.empty(n_frames)
for frame, ts in enumerate(u.trajectory):
    box_length = ts.dimensions[2]
    box_z[frame] = box_length
    # mean z of each group, with the four chains placed in the periodic image of the first
    group_z = {}
    for key, group in groups.items():
        z_values = group.positions[:, 2]
        shift = z_values - z_values[0]
        shift -= box_length * np.round(shift / box_length)
        group_z[key] = z_values[0] + shift.mean()
    relative = ions.positions[:, 2] - group_z["reference"]
    ion_z[frame] = relative - box_length * np.round(relative / box_length)
    relative = group_z["upper"] - group_z["reference"]
    upper_z[frame] = relative - box_length * np.round(relative / box_length)
    relative = group_z["lower"] - group_z["reference"]
    lower_z[frame] = relative - box_length * np.round(relative / box_length)
time_step = u.trajectory.dt / 1000           # ns
print(f"{n_frames} frames, upper plane {upper_z.mean():+.2f} A, lower plane {lower_z.mean():+.2f} A")

In [ ]:
n_inward = 0
n_outward = 0
for ion in range(ion_z.shape[1]):
    z_values = ion_z[:, ion]
    state = "below" if z_values[0] < lower_z[0] else ("above" if z_values[0] >= upper_z[0] else None)
    for frame in range(1, n_frames):
        if abs(z_values[frame] - z_values[frame - 1]) > box_z[frame] / 2:      # jump across the periodic boundary
            state = "below" if z_values[frame] < lower_z[frame] else ("above" if z_values[frame] >= upper_z[frame] else None)
            continue
        if z_values[frame] < lower_z[frame]:
            if state == "above" and frame * time_step >= start_time:
                n_inward += 1
            state = "below"
        elif z_values[frame] >= upper_z[frame]:
            if state == "below" and frame * time_step >= start_time:
                n_outward += 1
            state = "above"

net_events = n_inward - n_outward if direction == "in" else n_outward - n_inward
start_frame = int(round(start_time / time_step))
window = n_frames * time_step - start_time                         # ns
voltage = field_strength * box_z[start_frame:].mean() / 10        # V
conductance = net_events * ion_charge * 1.602176634e-19 / (window * 1e-9) / voltage * 1e12   # pS
print(f"inward {n_inward}, outward {n_outward}, net {net_events} in {window:.0f} ns, "
      f"V = {voltage * 1000:.0f} mV, g = {conductance:.1f} pS")